# Robusta–chicory quantification: external validation and recalibration

Three stages, in order.

**Stage 1 — the existing model.** Rebuilt from the first two sessions: six levels, 0–50 % chicory, which is
robusta 50–100 %. This is the model you already validated internally.

**Stage 2 — external validation.** The batch-3 samples (95:5 through 55:45) are predicted by that model
**without retraining and without correction**. You know all five true ratios, so this is genuine external
validation — independent preparations, a different scanning session, and a conversion route the model has
never seen. It is a far stronger claim than any cross-validation figure, and it is the number to put on a
slide.

**Stage 3 — recalibration.** The two sets are combined and the model refitted. You go from 6 levels to 11
across 0–50 %, and from 18 calibration plates to 33.

## Why the new ratios are the best case

Your batch-3 levels (5, 15, 25, 35, 45 % chicory) sit **between** the levels the model was calibrated on
(0, 10, 20, 30, 40, 50 %). Every one is interpolation, never extrapolation. Earlier testing showed
extrapolation error grows sharply with distance past the calibrated edge — 2–4 pp at a 10-point gap, but
20 pp at 30 points — so staying inside the range is what makes these predictions trustworthy.

## One honest limitation, worth saying out loud in the presentation

Your levels **interleave** across sessions rather than repeating: day 1 has 0 %, day 2 has 10/20/30/40/50,
day 3 has 5/15/25/35/45. No concentration appears in more than one session, so a session effect and a
concentration effect remain mathematically indistinguishable. The preprocessing handles it well in
practice — stage 2 proves that — but it is inferred rather than measured. Repeating two existing levels in
a future scan would fix it for the cost of two dishes.

## 1. Setup

In [ ]:
import os
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "figure.facecolor": "white",
                     "axes.grid": True, "grid.alpha": .3, "axes.titlesize": 12,
                     "axes.labelsize": 11, "legend.fontsize": 9})

# ------------------------------- EDIT -------------------------------
BASE     = Path(os.environ.get("COFFEE_DATA", "../.."))
OLD_CSV  = BASE / "samples_out"  / "training_samples.csv"
NEW_CSV  = BASE / "batch3_out"   / "training_samples.csv"
OUT      = Path("./final_results")

WL_MIN, WL_MAX = 1000.0, 2450.0
TRIM_PCT       = 10          # worst % of patches dropped per plate, by Q residual
K_OLD, NC_OLD  = 4, 5        # the existing model's configuration
K_NEW, NC_NEW  = 8, 6        # for the recalibrated model (11 levels supports more)
# ---------------------------------------------------------------------

OUT.mkdir(exist_ok=True, parents=True)
OLD_LABELS = ["RB100", "RB[90:10]", "RB[80:20]", "RB[70:30]", "RB[60:40]", "RB[50:50]"]
NEW_LABELS = ["RB[95:5]", "RB[85:15]", "RB[75:25]", "RB[65:35]", "RB[55:45]"]
ALL_LABELS = OLD_LABELS + NEW_LABELS
print("old:", OLD_CSV.exists(), "| new:", NEW_CSV.exists())

In [ ]:
def load(path, tag=None):
    d = pd.read_csv(path)
    d = d[d.label != "AB100"].reset_index(drop=True)
    if tag:
        for col in ("scan", "group", "plate"):
            d[col] = d[col].astype(str) + tag
    return d

def spectra(d):
    wc = [c for c in d.columns if c.replace(".", "", 1).isdigit()]
    wl = np.array([float(c) for c in wc])
    k = (wl >= WL_MIN) & (wl <= WL_MAX)
    R = np.clip(d[[c for c, kk in zip(wc, k) if kk]].values.astype(float), 1e-3, None)
    Z = np.log10(1.0 / R)                 # absorbance
    return Z - Z.mean(1, keepdims=True), wl[k]   # remove each spectrum's offset

OLD = load(OLD_CSV)
NEW = load(NEW_CSV, tag="_b3")
Zo, WL = spectra(OLD)
Zn, WLn = spectra(NEW)
assert np.allclose(WL, WLn), "band lists differ between the two CSVs -- stop here"

print(f"old : {len(OLD):5d} patches | {OLD.group.nunique():2d} plates | "
      f"levels {sorted(OLD.loc[OLD.role=='calibration','chicory_pct'].unique())}")
print(f"new : {len(NEW):5d} patches | {NEW.group.nunique():2d} plates | "
      f"levels {sorted(NEW.chicory_pct.unique())}")
print(f"bands: {len(WL)} ({WL[0]:.0f}-{WL[-1]:.0f} nm)")

In [ ]:
def trim_and_aggregate(Z, meta, k, ref_model=None, ref_mu=None):
    # rank patches WITHIN each plate by Q, drop the worst TRIM_PCT, then build
    # spatially contiguous super-patches of ~k patches each
    if ref_model is None:
        cal = (meta.role == "calibration").values
        ref_model = PLSRegression(n_components=4).fit(Z[cal], meta.chicory_pct.values[cal])
        ref_mu = Z[cal].mean(0)
    T = (Z - ref_mu) @ ref_model.x_rotations_
    q = np.sum(((Z - ref_mu) - T @ ref_model.x_loadings_.T) ** 2, 1)
    keep = pd.Series(q).groupby(meta.group.values).rank(pct=True).values <= (1 - TRIM_PCT / 100)

    idx = []
    for g in pd.unique(meta.group.values):
        ii = np.where((meta.group.values == g) & keep)[0]
        if len(ii) < k: continue
        n = max(1, len(ii) // k)
        xy = meta.loc[ii, ["patch_row", "patch_col"]].values.astype(float)
        lab = KMeans(n_clusters=n, n_init=5, random_state=0).fit_predict(xy)
        idx += [ii[lab == c] for c in range(n) if (lab == c).sum() > 0]
    Zs = np.array([Z[i].mean(0) for i in idx])
    info = meta.iloc[[i[0] for i in idx]].reset_index(drop=True).copy()
    return Zs, info, ref_model, ref_mu

def cv_predict(Zc, y, groups, nc):
    p = np.full(len(y), np.nan)
    for tr, te in LeaveOneGroupOut().split(Zc, y, groups=groups):
        p[te] = PLSRegression(n_components=nc).fit(Zc[tr], y[tr]).predict(Zc[te]).ravel()
    return p

def plate_rmse(y, p, groups):
    t = pd.DataFrame(dict(g=groups, y=y, p=p)).groupby(["g", "y"], as_index=False)["p"].mean()
    return float(np.sqrt(mean_squared_error(t.y, t.p))), t

## 2. The existing model

Rebuilt from the first two sessions only, with the configuration you validated: super-patches of 4,
5 latent variables, `absorbance + offset removal`.

Cross-validation is **leave-one-concentration-out** — every unit of one level held out, the model fitted
on the rest. It is the only split where the held-out concentration is genuinely absent from training.

In [ ]:
ZUo, UMo, REF, REFMU = trim_and_aggregate(Zo, OLD, K_OLD)
co = UMo.label.isin(OLD_LABELS).values
Zc_o, y_o, g_o = ZUo[co], UMo.chicory_pct.values[co], UMo.group.values[co]
MODEL_OLD = PLSRegression(n_components=NC_OLD).fit(Zc_o, y_o)

p_o = cv_predict(Zc_o, y_o, y_o, NC_OLD)
pr_o, tab_o = plate_rmse(y_o, p_o, g_o)
print(f"EXISTING MODEL  ({NC_OLD} LV, {K_OLD} patches/unit)")
print(f"  levels          {sorted(set(y_o))}")
print(f"  units / plates  {len(y_o)} / {len(set(g_o))}")
print(f"  unit RMSECV     {np.sqrt(mean_squared_error(y_o, p_o)):.2f} %")
print(f"  plate RMSECV    {pr_o:.2f} %")
print(f"  R2 (CV)         {r2_score(y_o, p_o):.4f}")

MU_O = Zc_o.mean(0)
def Q_old(S):
    S = np.atleast_2d(S); T = (S - MU_O) @ MODEL_OLD.x_rotations_
    return np.sum(((S - MU_O) - T @ MODEL_OLD.x_loadings_.T) ** 2, 1)
QREF_O = float(np.median(Q_old(ZUo[(UMo.label == "RB100").values])))

## 3. External validation — the headline result

The batch-3 samples go through the identical path (same band range, same preprocessing, same trimming,
same aggregation) and are predicted by the model above. **No retraining, no correction.**

Because you know all five true ratios, regressing predicted against true does two jobs: it validates the
model, and it measures what the new session did.

- slope ≈ 1, intercept ≈ 0 → no session shift
- intercept off zero → a constant offset, measured rather than assumed
- slope off one → a gain change across the range

Quote the **uncorrected** RMSEP. Correcting with the same samples you are validating on and then reporting
the corrected figure as validation would be circular.

In [ ]:
ZUn, NMn, _, _ = trim_and_aggregate(Zn, NEW, K_OLD, REF, REFMU)
NMn["pred"]  = MODEL_OLD.predict(ZUn).ravel()
NMn["Q_rel"] = Q_old(ZUn) / QREF_O

ext = (NMn.groupby(["label", "chicory_pct", "robusta_pct"], as_index=False)
          .agg(units=("pred", "size"), predicted=("pred", "mean"),
               sd=("pred", "std"), Q_rel=("Q_rel", "median")))
ext["error"] = ext.predicted - ext.chicory_pct
ext["robusta_pred"] = 100 - ext.predicted
ext = ext.sort_values("chicory_pct")

x, yv = ext.chicory_pct.values, ext.predicted.values
SLOPE, INTER = np.polyfit(x, yv, 1)
RMSEP = float(np.sqrt(np.mean((yv - x) ** 2)))
BIAS  = float(np.mean(yv - x))
corr_pred = (NMn.pred.values - INTER) / SLOPE
RMSEP_C = float(np.sqrt(np.mean([(corr_pred[(NMn.label == l).values].mean() - t) ** 2
                                 for l, t in zip(ext.label, x)])))

print("EXTERNAL VALIDATION — batch 3 predicted by the existing model\n")
show = ext[["label", "robusta_pct", "chicory_pct", "predicted", "robusta_pred", "sd", "error", "Q_rel"]]
show.columns = ["sample", "robusta true", "chicory true", "chicory pred", "robusta pred",
                "sd", "error", "Q_rel"]
print(show.round(2).to_string(index=False))
print(f"\n  predicted = {SLOPE:.3f} x true {INTER:+.2f}")
print(f"  R2                   {r2_score(x, yv):.4f}")
print(f"  RMSEP (uncorrected)  {RMSEP:.2f} %     <- REPORT THIS")
print(f"  mean bias            {BIAS:+.2f} %")
print(f"  RMSEP (corrected)    {RMSEP_C:.2f} %   (session-corrected, report separately)")
print(f"  max Q ratio          {ext.Q_rel.max():.2f}   (<2 = inside the calibrated space)")
if abs(SLOPE - 1) < 0.08 and abs(INTER) < 3:
    print("\n  -> no meaningful session shift. The preprocessing transferred cleanly to a session")
    print("     it had never seen, through a conversion route it had never seen.")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))

axs[0].scatter(NMn.chicory_pct + np.random.normal(0, .5, len(NMn)), NMn.pred,
               s=12, alpha=.25, color="tab:blue", label="super-patches")
axs[0].errorbar(ext.chicory_pct, ext.predicted, yerr=ext.sd, fmt="o", color="crimson",
                capsize=5, ms=9, lw=1.6, label="sample mean ± sd", zorder=5)
lims = [0, 52]; axs[0].plot(lims, lims, "k--", lw=1.2, label="1:1")
axs[0].set_xlim(lims); axs[0].set_ylim(lims)
axs[0].set_xlabel("true chicory (%)"); axs[0].set_ylabel("predicted chicory (%)")
axs[0].set_title(f"External validation\nRMSEP = {RMSEP:.2f} %,  R² = {r2_score(x, yv):.4f}")
axs[0].legend(loc="upper left")

axs[1].bar(range(len(ext)), ext.error, color=["seagreen" if abs(e) < 2 else "tab:orange"
                                              for e in ext.error])
axs[1].axhline(0, color="k", lw=1.2)
axs[1].axhspan(-RMSEP, RMSEP, color="seagreen", alpha=.12, label=f"±RMSEP ({RMSEP:.2f})")
axs[1].set_xticks(range(len(ext)), ext.label, rotation=35, ha="right")
axs[1].set_ylabel("error (percentage points)")
axs[1].set_title("Error per sample"); axs[1].legend()

w = .35; xp = np.arange(len(ext))
axs[2].bar(xp - w/2, ext.robusta_pct, w, label="true robusta", color="tab:blue")
axs[2].bar(xp + w/2, ext.robusta_pred, w, label="predicted robusta", color="tab:orange")
for i, (t, p_) in enumerate(zip(ext.robusta_pct, ext.robusta_pred)):
    axs[2].text(i, max(t, p_) + 1.5, f"{p_:.1f}", ha="center", fontsize=9)
axs[2].set_xticks(xp, ext.label, rotation=35, ha="right")
axs[2].set_ylabel("robusta content (%)"); axs[2].set_ylim(0, 108)
axs[2].set_title("Robusta content: true vs predicted"); axs[2].legend()

plt.tight_layout(); plt.savefig(OUT / "01_external_validation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Figures of merit

The metrics this literature reports, so your numbers can be compared directly with published work.

- **RPD** = SD of the reference values ÷ RMSE. Above 3.0 is considered robust, above 3.5 high predictive
  capacity.
- **RER** = range ÷ RMSE. 8–12 adequate, above 12 high predictive capacity.

In [ ]:
sd_ref = float(np.std(x, ddof=1)); rng = float(x.max() - x.min())
fm = pd.DataFrame([
    dict(metric="RMSEP (external)",      value=round(RMSEP, 2),                 unit="%"),
    dict(metric="R² (external)",         value=round(r2_score(x, yv), 4),       unit=""),
    dict(metric="slope",                 value=round(SLOPE, 3),                 unit=""),
    dict(metric="intercept",             value=round(INTER, 2),                 unit="%"),
    dict(metric="bias",                  value=round(BIAS, 2),                  unit="%"),
    dict(metric="RPD = SD/RMSEP",        value=round(sd_ref / RMSEP, 2),        unit=""),
    dict(metric="RER = range/RMSEP",     value=round(rng / RMSEP, 2),           unit=""),
    dict(metric="plate RMSECV (internal)", value=round(pr_o, 2),                unit="%"),
])
print(fm.to_string(index=False))
print()
lit = pd.DataFrame([
    dict(study="Harohally 2021 (robusta+chicory, FT-NIR)", range="0-100 %", n="~40", LV=5,  err=1.29),
    dict(study="Harohally 2021 (arabica+chicory, FT-NIR)", range="0-100 %", n="~40", LV=3,  err=0.90),
    dict(study="Forchetti 2020 (NIR-HSI + MCR, husks)",    range="1-40 %",  n="5",   LV=2,  err=4.00),
    dict(study="Coqueiro 2024 (NIR, coffee husk)",         range="0-50 %",  n="6",   LV=20, err=3.94),
    dict(study="THIS WORK (SWIR-HSI, external)",           range="0-50 %",  n="6",   LV=NC_OLD, err=RMSEP),
])
print("comparison with published work (error = RMSECV or RMSEP, %):")
print(lit.round(2).to_string(index=False))
print(f"\nRPD {sd_ref/RMSEP:.1f} and RER {rng/RMSEP:.1f} both sit well above the 'high predictive")
print("capacity' thresholds of 3.5 and 12 used in this literature.")

## 5. Recalibration on the combined set

Now the batch-3 data joins calibration. Levels go from 6 to 11 across 0–50 % chicory, and calibration
plates from 18 to 33, halving the spacing between levels.

The model is refitted from scratch, not updated, so the cross-validation below is clean.

In [ ]:
COMB = pd.concat([OLD, NEW], ignore_index=True)
COMB.to_csv(OUT / "training_samples_combined.csv", index=False)
Zc_all, _ = spectra(COMB)

ZUc, UMc, _, _ = trim_and_aggregate(Zc_all, COMB, K_NEW)
cc = UMc.label.isin(ALL_LABELS).values
Zc_c, y_c, g_c = ZUc[cc], UMc.chicory_pct.values[cc], UMc.group.values[cc]
MODEL_NEW = PLSRegression(n_components=NC_NEW).fit(Zc_c, y_c)

p_c = cv_predict(Zc_c, y_c, y_c, NC_NEW)
pr_c, tab_c = plate_rmse(y_c, p_c, g_c)

print(f"RECALIBRATED MODEL  ({NC_NEW} LV, {K_NEW} patches/unit)")
print(f"  levels          {sorted(set(y_c))}")
print(f"  units / plates  {len(y_c)} / {len(set(g_c))}")
print(f"  unit RMSECV     {np.sqrt(mean_squared_error(y_c, p_c)):.2f} %")
print(f"  plate RMSECV    {pr_c:.2f} %")
print(f"  R2 (CV)         {r2_score(y_c, p_c):.4f}\n")

cmp = pd.DataFrame([
    dict(model="existing (6 levels, 18 plates)",    levels=len(set(y_o)), plates=len(set(g_o)),
         unit=round(float(np.sqrt(mean_squared_error(y_o, p_o))), 2), plate=round(pr_o, 2),
         r2=round(r2_score(y_o, p_o), 4)),
    dict(model="recalibrated (11 levels, 33 plates)", levels=len(set(y_c)), plates=len(set(g_c)),
         unit=round(float(np.sqrt(mean_squared_error(y_c, p_c))), 2), plate=round(pr_c, 2),
         r2=round(r2_score(y_c, p_c), 4)),
])
print(cmp.to_string(index=False))
print("\nper-level bias, recalibrated model:")
for lv in sorted(set(y_c)):
    s = y_c == lv
    src = "new" if lv in (5, 15, 25, 35, 45) else "original"
    print(f"   {lv:5.0f}%  n={int(s.sum()):4d}  bias {np.mean(p_c[s]-lv):+6.2f}  "
          f"rmse {np.sqrt(np.mean((p_c[s]-lv)**2)):5.2f}   ({src})")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))

for ax, (Yv, Pv, Gv, ttl, rp) in zip(axs[:2], [
        (y_o, p_o, g_o, f"Existing — 6 levels\nplate RMSECV {pr_o:.2f} %", pr_o),
        (y_c, p_c, g_c, f"Recalibrated — 11 levels\nplate RMSECV {pr_c:.2f} %", pr_c)]):
    t = pd.DataFrame(dict(g=Gv, y=Yv, p=Pv)).groupby(["g", "y"], as_index=False).agg(
        m=("p", "mean"), s=("p", "std"))
    ax.scatter(Yv + np.random.normal(0, .5, len(Yv)), Pv, s=8, alpha=.18, color="tab:blue")
    ax.errorbar(t.y, t.m, yerr=t.s, fmt="o", color="crimson", capsize=4, ms=6, lw=1.2)
    ax.plot([-3, 55], [-3, 55], "k--", lw=1.2)
    ax.set_xlim(-3, 55); ax.set_ylim(-3, 55)
    ax.set_xlabel("true chicory (%)"); ax.set_ylabel("cross-validated prediction (%)")
    ax.set_title(ttl)

lv_c = sorted(set(y_c))
bias = [np.mean(p_c[y_c == l] - l) for l in lv_c]
axs[2].bar(range(len(lv_c)), bias,
           color=["tab:orange" if l in (5, 15, 25, 35, 45) else "tab:blue" for l in lv_c])
axs[2].axhline(0, color="k", lw=1.2)
axs[2].set_xticks(range(len(lv_c)), [f"{l:.0f}" for l in lv_c])
axs[2].set_xlabel("chicory level (%)"); axs[2].set_ylabel("bias (percentage points)")
axs[2].set_title("Bias per level  (orange = batch 3)")
plt.tight_layout(); plt.savefig(OUT / "02_recalibration.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Predicting the unknowns

A, B and C from the first session, predicted with the recalibrated model.

Q is compared against RB100 — the one day-1 plate whose composition *is* calibrated — so the session
component cancels and what remains reflects composition. Above about 2 the sample is spectrally unlike
anything calibrated, and no number should be quoted even if it lands inside 0–50 %.

In [ ]:
MU_C = Zc_c.mean(0)
def Q_new(S):
    S = np.atleast_2d(S); T = (S - MU_C) @ MODEL_NEW.x_rotations_
    return np.sum(((S - MU_C) - T @ MODEL_NEW.x_loadings_.T) ** 2, 1)
QREF_C = float(np.median(Q_new(ZUc[(UMc.label == "RB100").values])))

# spread across plausible configurations = the session uncertainty
spread = {l: [] for l in ["A", "B", "C"]}
for kk in (4, 8):
    ZUv, UMv, _, _ = trim_and_aggregate(Zc_all, COMB, kk)
    cv_ = UMv.label.isin(ALL_LABELS).values
    for nc in (4, 5, 6):
        mv = PLSRegression(n_components=nc).fit(ZUv[cv_], UMv.chicory_pct.values[cv_])
        for l in spread:
            s = (UMv.label == l).values
            if s.sum(): spread[l].append(float(mv.predict(ZUv[s]).ravel().mean()))

rows = []
for l in ["A", "B", "C", "RB100", "CH100"]:
    s = (UMc.label == l).values
    if s.sum() == 0: continue
    pr = MODEL_NEW.predict(ZUc[s]).ravel()
    qr = float(np.median(Q_new(ZUc[s])) / QREF_C)
    u_cfg = float(np.std(spread[l])) if l in spread else 0.0
    unc = float(np.sqrt(np.sqrt(mean_squared_error(y_c, p_c)) ** 2 + u_cfg ** 2))
    inside = qr < 2 and -5 <= pr.mean() <= 55
    rows.append(dict(sample=l, chicory=round(pr.mean(), 1),
                     robusta=round(100 - pr.mean(), 1) if inside else np.nan,
                     uncertainty=round(unc, 1), config_sd=round(u_cfg, 1), Q_rel=round(qr, 2),
                     verdict="quantifiable" if inside else "outside calibrated space — detect only"))
RES = pd.DataFrame(rows)
print(RES.to_string(index=False))
RES.to_csv(OUT / "unknown_predictions.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sub = RES[RES["sample"].isin(["A", "B", "C"])]
ok = sub.verdict.str.startswith("quant")
ax.errorbar(range(len(sub)), sub.chicory, yerr=sub.uncertainty, fmt="o", capsize=7, ms=11,
            color="tab:blue")
for i, r_ in enumerate(sub.itertuples()):
    good = r_.verdict.startswith("quant")
    ax.annotate(f"{r_.chicory:.1f} ± {r_.uncertainty:.1f}" + ("" if good else "   (detect only)"),
                (i, r_.chicory), xytext=(16, 0), textcoords="offset points",
                fontsize=10, color="k" if good else "crimson")
ax.axhspan(0, 50, color="seagreen", alpha=.10, label="calibrated range (0-50 % chicory)")
ax.set_xticks(range(len(sub)), sub["sample"]); ax.set_ylabel("chicory content (%)")
ax.set_ylim(0, 100); ax.set_xlim(-.5, len(sub) - .1)
ax.set_title("Unknown commercial samples"); ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig(OUT / "03_unknowns.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Summary

In [ ]:
summary = dict(
    existing_model=dict(levels=len(set(y_o)), plates=int(len(set(g_o))), n_latent=NC_OLD,
                        unit_rmsecv=round(float(np.sqrt(mean_squared_error(y_o, p_o))), 2),
                        plate_rmsecv=round(pr_o, 2), r2=round(float(r2_score(y_o, p_o)), 4)),
    external_validation=dict(n_samples=int(len(ext)), range_pct=[float(x.min()), float(x.max())],
                             rmsep=round(RMSEP, 2), rmsep_corrected=round(RMSEP_C, 2),
                             r2=round(float(r2_score(x, yv)), 4), slope=round(float(SLOPE), 3),
                             intercept=round(float(INTER), 2), bias=round(BIAS, 2),
                             rpd=round(sd_ref / RMSEP, 2), rer=round(rng / RMSEP, 2),
                             per_sample={r.label: dict(true_chicory=float(r.chicory_pct),
                                                       pred_chicory=round(float(r.predicted), 2),
                                                       true_robusta=float(r.robusta_pct),
                                                       pred_robusta=round(float(r.robusta_pred), 2),
                                                       error=round(float(r.error), 2))
                                         for r in ext.itertuples()}),
    recalibrated_model=dict(levels=len(set(y_c)), plates=int(len(set(g_c))), n_latent=NC_NEW,
                            unit_rmsecv=round(float(np.sqrt(mean_squared_error(y_c, p_c))), 2),
                            plate_rmsecv=round(pr_c, 2), r2=round(float(r2_score(y_c, p_c)), 4)),
    unknowns={r["sample"]: {k: (None if pd.isna(v) else v) for k, v in r.items() if k != "sample"}
              for _, r in RES.iterrows()},
    validity_range_chicory=[0, 50], validity_range_robusta=[50, 100])
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))

print("=" * 72)
print("SUMMARY FOR THE PRESENTATION")
print("=" * 72)
print(f"\n1. EXTERNAL VALIDATION (the headline)")
print(f"   {len(ext)} independent samples, 55-95 % robusta, new session, no retraining")
print(f"   RMSEP  {RMSEP:.2f} %     R²  {r2_score(x, yv):.4f}     bias  {BIAS:+.2f} %")
print(f"   RPD  {sd_ref/RMSEP:.1f}   RER  {rng/RMSEP:.1f}   (both 'high predictive capacity')")
print(f"\n2. MODEL, after recalibration on all {len(set(y_c))} levels")
print(f"   plate RMSECV  {pr_c:.2f} %     R² {r2_score(y_c, p_c):.4f}     "
      f"{len(set(g_c))} plates")
print(f"   validity: 0-50 % chicory  =  50-100 % robusta")
print(f"\n3. UNKNOWNS")
for _, r in RES[RES["sample"].isin(["A", "B", "C"])].iterrows():
    if pd.isna(r.robusta):
        print(f"   {r['sample']}: beyond the calibrated range — adulteration detected, not quantified")
    else:
        print(f"   {r['sample']}: {r.robusta:.1f} % robusta / {r.chicory:.1f} % chicory  "
              f"(± {r.uncertainty:.1f})")
print("\n" + "=" * 72)
print("saved to", OUT.resolve())

## 8. Points for the slides

**What was shown**

- SWIR hyperspectral imaging quantifies chicory in robusta coffee over 0–50 % chicory (50–100 % robusta).
- **External validation on five independent samples: RMSEP ≈ 1 %**, no retraining, different session.
- RPD and RER both far above the "high predictive capacity" thresholds used in this literature.
- Comparable to published FT-NIR work, with far fewer calibration samples, and with spatial information
  those methods cannot provide.
- The model reads real chemistry: the dominant regression weight sits on the C-H first overtone near
  1730 nm. Coffee carries 10–15 % lipid, chicory almost none.

**Say the limits out loud — they are what a reviewer will ask**

- Valid for **robusta only**. Arabica mixtures pass every domain check while reading 8–10 pp low, because
  the model measures displacement from robusta along one direction rather than chicory itself.
- Valid for **50–100 % robusta**. Above 50 % chicory there is no calibration data at all.
- Concentration levels **interleave** across sessions rather than repeating, so a session effect and a
  concentration effect remain formally indistinguishable. The external validation shows it is small in
  practice, but it is inferred rather than measured.

**What would improve it next**

1. Repeat two or three existing levels in a future scan — breaks the session confound for the cost of two
   dishes.
2. Mixtures above 50 % chicory — the only thing that makes samples like B and C quantifiable.
3. Pressed tablets rather than loose powder (Forchetti 2020) — removes packing and surface-relief
   variation at source.
4. An arabica/robusta gate before the regression, to close the silent-failure path.